# Get env along the obs trajectory

In [1]:
cd ~/work/tutorials/sources/OpenDrift/openberg_july2026

/home/jovyan/work/tutorials/sources/OpenDrift/openberg_july2026


In [2]:
import glob
from src.utils import *
from openberg_july2026.src2.utils0 import *
from openberg_july2026.src2.utils2 import *

In [3]:
ib = 'iceberg2011w2'
cmems_l = ['topaz4','glorys','windglophyre']
# cmems_l = ['topaz4','topaz5','topaz5-ensemble','topaz6','glorys','glophyanfcH','glophyanfcD','windglophynrt']
sim_l_contain, sim_l_notcontain = [ib],['idx',]

#### Preload and calc

#### Create file

In [4]:
out = xr.Dataset()
                # coords={'time_obs':obs.time.data,
                #         'cmems_id':[env_dict[cm] for cm in cmems_l],}
                         # 'time_sim':np.arange(dict_analysis_period[ib]['full'].start,dict_analysis_period[ib]['full'].stop,dtype='datetime64[h]'),
                         # 'time_sim':np.arange(obs_seed.time.data[0],obs_seed.time.data[-1]+
                                              # np.timedelta64(int(str(obs.seed_freq.data)[:-1]),str(obs.seed_freq.data)[-1]),dtype='datetime64[h]'),
                         # 'sim_id':[f.split('/')[-1].replace('.nc','') for f in file_l],
                         # 'sim_trajectory':np.arange(obs_seed.time.size*11)},
                # data_vars={v:(('time_obs','cmems_id'),np.full((obs.time.data.size,len(cmems_l)),np.nan)) for v in ['x_sea_water_velocity','y_sea_water_velocity',
                # "sea_ice_area_fraction","sea_ice_x_velocity","sea_ice_y_velocity"]}

#### Observations

In [5]:
#read obs
fname = 'merged_obs10'#'merged_obs_iceberg2026e_1D'
with xr.open_dataset('./input/%s.nc'%fname) as ds:
    obs = ds.sel(iceberg=ib,trajectory_id=ib) 
    obs_seed = obs.isel(time=(obs.seed_idx==1))

In [6]:
obs_sub_kin = compute_drift_kinematics(obs)
tl = [str(t)[:16] for t in obs.time.data]

In [7]:
#add obs velocity
out = out.assign_coords(time_obs=("time_obs", obs.time.data))

out['iceberg_u_ms'] = (('time_obs'), obs_sub_kin.u_ms.values)
out['iceberg_v_ms'] = (('time_obs'), obs_sub_kin.v_ms.values)
out['iceberg_speed_ms'] = (('time_obs'), obs_sub_kin.speed_ms.values)
out['iceberg_direction_deg'] = (('time_obs'), obs_sub_kin.direction_deg.values)

In [8]:
out

<xarray.Dataset> Size: 54kB
Dimensions:                (time_obs: 1345)
Coordinates:
  * time_obs               (time_obs) datetime64[ns] 11kB 2011-09-26T15:05:00...
Data variables:
    iceberg_u_ms           (time_obs) float64 11kB nan -0.1245 ... -0.0006718
    iceberg_v_ms           (time_obs) float64 11kB nan -0.03814 ... 0.0003488
    iceberg_speed_ms       (time_obs) float64 11kB nan 0.1302 ... 0.0 0.000757
    iceberg_direction_deg  (time_obs) float64 11kB nan 253.0 238.8 ... 0.0 297.4

#### Simulations

In [9]:
#simulation list
file_l = glob.glob("./results/%s/*.nc"%ib)
file_l = [f for f in file_l if all(ff in f for ff in sim_l_contain)]
file_l = [f for f in file_l if all(ff not in f for ff in sim_l_notcontain)]
print(file_l)

['./results/iceberg2011w2/iceberg2011w2_gebco_glorys_windglophyre_newcoefmix_joined.nc', './results/iceberg2011w2/iceberg2011w2_gebco_topaz4_windglophyre_newcoefmix_joined.nc']


In [10]:
#load sim
try: #if not merged
    sim_l = [str(f.split('/')[-1][:-3]) for f in file_l] 
    simall = xr.open_mfdataset(file_l,combine='nested',concat_dim="run")
    simall = simall.assign_coords(run=sim_l) #if not merged yet
except:     #when merged lready
    simall = xr.open_mfdataset(file_l)
    sim_l = [str(f) for f in simall.run]

In [11]:
# add sim velocity
out = out.assign_coords(time_sim=("time_sim", simall.time.data))
out = out.assign_coords(sim_id=("sim_id", simall.run.data))
out = out.assign_coords(sim_trajectory=("sim_trajectory", simall.trajectory.data))

out['iceb_x_velocity'] = (('sim_id','sim_trajectory','time_sim'), simall.iceb_x_velocity.values)
out['iceb_y_velocity'] = (('sim_id','sim_trajectory','time_sim'), simall.iceb_y_velocity.values)

In [12]:
out

<xarray.Dataset> Size: 2MB
Dimensions:                (time_obs: 1345, time_sim: 960, sim_id: 2,
                            sim_trajectory: 110)
Coordinates:
  * time_obs               (time_obs) datetime64[ns] 11kB 2011-09-26T15:05:00...
  * time_sim               (time_sim) datetime64[ns] 8kB 2011-09-26T15:05:00 ...
  * sim_id                 (sim_id) <U57 456B 'iceberg2011w2_gebco_glorys_win...
  * sim_trajectory         (sim_trajectory) int32 440B 0 1 2 3 ... 107 108 109
Data variables:
    iceberg_u_ms           (time_obs) float64 11kB nan -0.1245 ... -0.0006718
    iceberg_v_ms           (time_obs) float64 11kB nan -0.03814 ... 0.0003488
    iceberg_speed_ms       (time_obs) float64 11kB nan 0.1302 ... 0.0 0.000757
    iceberg_direction_deg  (time_obs) float64 11kB nan 253.0 238.8 ... 0.0 297.4
    iceb_x_velocity        (sim_id, sim_trajectory, time_sim) float32 845kB 0...
    iceb_y_velocity        (sim_id, sim_trajectory, time_sim) float32 845kB 0...

#### Cmems along obs

In [13]:
# add env along obs
out = out.assign_coords(cmems_id=("cmems_id", cmems_l))#[env_dict[cm] if cm!='topaz5-ensemble' else env_dict[cm][0] for cm in cmems_l]))
    
for v in ['x_sea_water_velocity','y_sea_water_velocity',"sea_ice_area_fraction","sea_ice_x_velocity","sea_ice_y_velocity","x_wind","y_wind"]:
    out[v] = (('time_obs','cmems_id'), np.full((obs.time.data.size,len(cmems_l)),np.nan))


In [14]:
out

<xarray.Dataset> Size: 2MB
Dimensions:                (time_obs: 1345, time_sim: 960, sim_id: 2,
                            sim_trajectory: 110, cmems_id: 3)
Coordinates:
  * time_obs               (time_obs) datetime64[ns] 11kB 2011-09-26T15:05:00...
  * time_sim               (time_sim) datetime64[ns] 8kB 2011-09-26T15:05:00 ...
  * sim_id                 (sim_id) <U57 456B 'iceberg2011w2_gebco_glorys_win...
  * sim_trajectory         (sim_trajectory) int32 440B 0 1 2 3 ... 107 108 109
  * cmems_id               (cmems_id) <U12 144B 'topaz4' 'glorys' 'windglophyre'
Data variables: (12/13)
    iceberg_u_ms           (time_obs) float64 11kB nan -0.1245 ... -0.0006718
    iceberg_v_ms           (time_obs) float64 11kB nan -0.03814 ... 0.0003488
    iceberg_speed_ms       (time_obs) float64 11kB nan 0.1302 ... 0.0 0.000757
    iceberg_direction_deg  (time_obs) float64 11kB nan 253.0 238.8 ... 0.0 297.4
    iceb_x_velocity        (sim_id, sim_trajectory, time_sim) float32 845kB 0...
    iceb_y_velocity        (sim_id, sim_trajectory, time_sim) float32 845kB 0...
    ...                     ...
    y_sea_water_velocity   (time_obs, cmems_id) float64 32kB nan nan ... nan nan
    sea_ice_area_fraction  (time_obs, cmems_id) float64 32kB nan nan ... nan nan
    sea_ice_x_velocity     (time_obs, cmems_id) float64 32kB nan nan ... nan nan
    sea_ice_y_velocity     (time_obs, cmems_id) float64 32kB nan nan ... nan nan
    x_wind                 (time_obs, cmems_id) float64 32kB nan nan ... nan nan
    y_wind                 (time_obs, cmems_id) float64 32kB nan nan ... nan nan

In [15]:
for cmems_id in cmems_l:
    
    if cmems_id in ['windglophyre','windglophynrt']:variables=['x_wind','y_wind']
    elif cmems_id in ['topaz6','glophyanfcH']: variables = ['x_sea_water_velocity','y_sea_water_velocity']
    elif cmems_id in ['nextsimre','nextsimanfc','glophyanfcD']: 
        variables = ["sea_ice_area_fraction",
                "sea_ice_x_velocity",
                "sea_ice_y_velocity"]
    else:
        variables = ['x_sea_water_velocity',
                'y_sea_water_velocity',
                "sea_ice_area_fraction",
                "sea_ice_x_velocity",
                "sea_ice_y_velocity"]
        
    # cmems_id = env_dict[cmems_id]
    print(cmems_id,variables)
    
    if str(env_dict[cmems_id]).startswith('cmems_'):
    
        from opendrift.readers import reader_copernicusmarine
        reader = reader_copernicusmarine.Reader(
            dataset_id=env_dict[cmems_id])
    elif cmems_id=='topaz5-ensemble':
        ds_env = xr.open_mfdataset(env_dict[cmems_id],
                    concat_dim=xr.DataArray(range(len(env_dict[cmems_id])), dims='member', name='member',attrs={'standard_name': 'realization'}),
                    combine='nested', data_vars='all', coords='all', chunks={'time': 1}) #Solution from KF!
        from opendrift.readers import reader_netCDF_CF_generic
        reader = reader_netCDF_CF_generic.Reader(ds_env)
        del ds_env
    else:
        from opendrift.readers import reader_netCDF_CF_generic
        reader = reader_netCDF_CF_generic.Reader(
            env_dict[cmems_id])
        
    try:z0 = min(0.0, np.max(reader.z))
    except: z0=0
    
    for i,t in enumerate(tl):
        # print('%s/%s'%(i,len(tl)))
        env, profiles = reader.get_variables_interpolated(
            variables=variables,
            lon=obs.lon[i],
            lat=obs.lat[i],
            time=np.datetime64(t),
            z=z0
        )
        for v in variables:
            out[v].loc[{'time_obs':t,'cmems_id':cmems_id}] = env[v].data[0]

topaz4 ['x_sea_water_velocity', 'y_sea_water_velocity', 'sea_ice_area_fraction', 'sea_ice_x_velocity', 'sea_ice_y_velocity']


INFO - 2026-10-08T15:06:47Z - Selected dataset version: "202506"
INFO - 2026-10-08T15:06:47Z - Selected dataset part: "default"


glorys ['x_sea_water_velocity', 'y_sea_water_velocity', 'sea_ice_area_fraction', 'sea_ice_x_velocity', 'sea_ice_y_velocity']


INFO - 2026-10-08T15:07:15Z - Selected dataset version: "202311"
INFO - 2026-10-08T15:07:15Z - Selected dataset part: "default"
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - returning
Only NaNs input to linearNDFast - re

windglophyre ['x_wind', 'y_wind']


INFO - 2026-10-08T15:07:40Z - Selected dataset version: "202211"
INFO - 2026-10-08T15:07:40Z - Selected dataset part: "default"


#### Checks

In [16]:
out

<xarray.Dataset> Size: 2MB
Dimensions:                (time_obs: 1345, time_sim: 960, sim_id: 2,
                            sim_trajectory: 110, cmems_id: 3)
Coordinates:
  * time_obs               (time_obs) datetime64[ns] 11kB 2011-09-26T15:05:00...
  * time_sim               (time_sim) datetime64[ns] 8kB 2011-09-26T15:05:00 ...
  * sim_id                 (sim_id) <U57 456B 'iceberg2011w2_gebco_glorys_win...
  * sim_trajectory         (sim_trajectory) int32 440B 0 1 2 3 ... 107 108 109
  * cmems_id               (cmems_id) <U12 144B 'topaz4' 'glorys' 'windglophyre'
Data variables: (12/13)
    iceberg_u_ms           (time_obs) float64 11kB nan -0.1245 ... -0.0006718
    iceberg_v_ms           (time_obs) float64 11kB nan -0.03814 ... 0.0003488
    iceberg_speed_ms       (time_obs) float64 11kB nan 0.1302 ... 0.0 0.000757
    iceberg_direction_deg  (time_obs) float64 11kB nan 253.0 238.8 ... 0.0 297.4
    iceb_x_velocity        (sim_id, sim_trajectory, time_sim) float32 845kB 0...
    iceb_y_velocity        (sim_id, sim_trajectory, time_sim) float32 845kB 0...
    ...                     ...
    y_sea_water_velocity   (time_obs, cmems_id) float64 32kB -0.04925 ... nan
    sea_ice_area_fraction  (time_obs, cmems_id) float64 32kB 0.0 nan ... nan
    sea_ice_x_velocity     (time_obs, cmems_id) float64 32kB 0.0 0.0 ... nan
    sea_ice_y_velocity     (time_obs, cmems_id) float64 32kB 0.0 0.0 ... nan
    x_wind                 (time_obs, cmems_id) float64 32kB nan nan ... -3.145
    y_wind                 (time_obs, cmems_id) float64 32kB nan nan ... -3.165

In [17]:
#sim spd
# for v in ['iceb_x_velocity','iceb_y_velocity']:
#     out.plot.scatter(x='time_sim',y=v)
# plt.show()
# #obs spd
# for v in ['iceberg_u_ms','iceberg_v_ms','iceberg_speed_ms','iceberg_direction_deg']:
#     out.plot.scatter(x='time_obs',y=v)
# plt.show()
# # env spd
# for v in ['x_sea_water_velocity','y_sea_water_velocity','sea_ice_x_velocity','sea_ice_y_velocity','sea_ice_area_fraction']:
#     if not np.all(np.isnan(out[v])): out.plot.scatter(x='time_obs',y=v)
# plt.show()

#### Save

In [18]:
out.to_netcdf('./results/timeseries_coll/%s_timeseries_collection.nc'%ib)

#### Load and Check

In [19]:
# ib = 'iceberg2011w2'
# test = xr.open_dataset('./results/timeseries_coll/%s_timeseries_collection.nc'%ib)
# test.x_sea_water_velocity.plot()